# Local Agentic XAI-RAG — Scientific Paper Results Report

**Project:** `local_agentic_xai_rag` (Phase 3)  
**Objective:** Structured **advisory screening reports for human review** — not clinical diagnoses.  
**Runtime:** Ollama **`llama3` only** (generation, structured JSON, embeddings, claim verification).  
**Knowledge base:** Shared Phase 2 corpus (`explainability_augmented_rag/`).

This notebook consolidates all paper-ready outputs:
1. Architecture & configuration  
2. Acceptance gates (G1–G6)  
3. Gold-chunk retrieval evaluation (BM25 vs hybrid)  
4. Three-case advisory report evaluation (safety, claims)  
5. System ablation B1–B3 (+ 20-question benchmark subset)  
6. Phase 2 baseline comparison  
7. All six figures + summary tables  

> **Prerequisite:** Run the pipeline and eval scripts first (see README). This notebook **loads** pre-computed results from `results/` and `figures/`.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
%matplotlib inline

# Resolve project root (notebook may run from notebooks/ or project root)
ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if not (ROOT / "config.yaml").exists():
    raise FileNotFoundError("Run from local_agentic_xai_rag/ or notebooks/")

PHASE2 = ROOT.parent / "explainability_augmented_rag"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
PAPER = ROOT / "paper_outputs"
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
PALETTE = sns.color_palette("colorblind")

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

def load_csv(name: str) -> pd.DataFrame:
    p = RESULTS / name
    if not p.exists():
        raise FileNotFoundError(f"Missing {p} — run eval scripts first.")
    return pd.read_csv(p)

def load_jsonl(name: str) -> list[dict]:
    p = RESULTS / name
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

def show_figure(name: str, title: str = ""):
    p = FIGURES / name
    if not p.exists():
        print(f"Missing {p} — run: python -m local_agentic_xai_rag.visualization.build_all_figures")
        return
    img = mpimg.imread(p)
    fig, ax = plt.subplots(figsize=(10, 4 if "architecture" in name else 5))
    ax.imshow(img)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12, pad=8)
    plt.show()

print("Project root:", ROOT)
print("Phase 2:", PHASE2)
print("Ollama model:", CONFIG["ollama"]["model"])

## 1. Architecture

Lean human-in-the-loop pipeline: **2 agents** (evidence verification → advisory report) + **deterministic tools** (case format, hybrid retrieval, regex safety, claim verification).

```
Case + XAI card
  → CaseFormatterTool
  → Template queries (4 types)
  → HybridRetrieverTool (BM25 + llama3 dense, RRF)
  → Evidence verification agent (llama3 JSON)
  → Advisory report agent (llama3 JSON)
  → SafetyRulesTool (regex) + CitationCheckerTool (llama3)
  → optional 1 revision → final_narratives.jsonl → Expert review
```

**Not in v1:** query-planning agent, cross-encoder reranker, LLM-as-judge primary eval, full factorial ablations.

In [ ]:
show_figure("fig01_architecture.png", "Figure 1 — Human-in-the-loop advisory pipeline")

## 2. Corpus, agents & configuration (Table 1)

In [ ]:
chunks_path = PHASE2 / "data" / "processed_chunks.csv"
chunks = pd.read_csv(chunks_path)
n_docs = chunks["doc_id"].nunique() if "doc_id" in chunks.columns else "—"

table_corpus = pd.DataFrame([
    {"Component": "PDF corpus (Phase 2)", "Value": f"{n_docs} documents"},
    {"Component": "Text chunks", "Value": f"{len(chunks):,}"},
    {"Component": "Case profiles", "Value": "3 (high / low / borderline risk)"},
    {"Component": "Gold-chunk benchmark", "Value": "100 questions"},
    {"Component": "Ollama model", "Value": CONFIG["ollama"]["model"] + " (chat + embed + verify)"},
    {"Component": "Retrieval default", "Value": f"hybrid BM25+llama3, top_k={CONFIG['retrieval']['top_k']}, rrf_k={CONFIG['retrieval']['rrf_k']}"},
    {"Component": "Agents", "Value": "evidence_verification_synthesis + advisory_report_generation"},
    {"Component": "Tools", "Value": "CaseFormatter, HybridRetriever, SafetyRules, CitationChecker"},
])
display(table_corpus.style.hide(axis="index").set_caption("Table 1 — System configuration"))

## 3. Acceptance gates (G1–G6)

In [ ]:
gates_path = RESULTS / "acceptance_gates.json"
if gates_path.exists():
    gates = json.loads(gates_path.read_text(encoding="utf-8"))
else:
    from local_agentic_xai_rag.evaluation.acceptance import run_gates
    gates = run_gates()

gate_rows = [
    ("G1", "Valid final_narratives.jsonl (3 cases, all schema fields)", gates.get("G1_valid_schema")),
    ("G2", f"Disclaimer ≥ 90% (observed {gates.get('G2_disclaimer_rate', 0):.0%})", gates.get("G2_pass")),
    ("G3", f"No unsupported/contradicted claims (bad={gates.get('G3_bad_claims', '?')})", gates.get("G3_pass")),
    ("G4", "Retrieval eval on 100 gold-chunk questions", gates.get("G4_retrieval_eval")),
    ("G5", "Ollama llama3 only (local)", gates.get("G5_ollama_local")),
    ("G6", "Results logged under results/", gates.get("G6_results_logged")),
]
df_gates = pd.DataFrame(gate_rows, columns=["Gate", "Criterion", "Pass"])
display(df_gates.assign(Pass=df_gates["Pass"].map({True: "✓", False: "✗", None: "?"})))
print(f"\nALL_PASS = {gates.get('ALL_PASS')}")

## 4. Retrieval evaluation — gold-chunk benchmark (100 questions)

Objective metrics: P@5, R@5, R@10, MRR, nDCG@10.  
**Phase 2 baselines:** BM25 P@5 = 0.128; hybrid P@5 = 0.088.

In [ ]:
retrieval_summary = load_csv("retrieval_eval_summary.csv")
retrieval_detail = load_csv("retrieval_eval_results.csv")

phase2_baselines = pd.DataFrame([
    {"retriever": "bm25 (Phase 2 baseline)", "precision@5": 0.128, "recall@5": np.nan, "recall@10": np.nan},
    {"retriever": "hybrid (Phase 2 baseline)", "precision@5": 0.088, "recall@5": np.nan, "recall@10": np.nan},
])

display(retrieval_summary.style.format({c: "{:.4f}" for c in retrieval_summary.columns if c != "retriever"})
        .set_caption("Table 2 — Retrieval metrics (this project)"))

fig, ax = plt.subplots(figsize=(8, 4))
metrics = ["precision@5", "recall@5", "recall@10", "mrr", "ndcg@10"]
retrieval_summary.set_index("retriever")[metrics].plot(kind="bar", ax=ax, color=PALETTE[:2])
ax.set_title("Retrieval: BM25 vs hybrid llama3")
ax.set_ylim(0, max(0.35, retrieval_summary[metrics].max().max() * 1.15))
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

show_figure("fig02_retrieval_bm25_vs_hybrid.png")
show_figure("fig06_benchmark_category_heatmap.png", "Recall@5 by benchmark category")

In [ ]:
# Per-category retrieval (from detailed results)
if "category" in retrieval_detail.columns:
    cat_summary = (
        retrieval_detail.groupby(["category", "retriever"])[["precision@5", "recall@5", "recall@10"]]
        .mean().round(4).reset_index()
    )
    display(cat_summary.head(20).style.set_caption("Retrieval by category (top rows)"))

## 5. Advisory report evaluation (3 cases)

Deterministic metrics: JSON schema validity, disclaimer/referral regex, claim support distribution.  
**Phase 2 baseline:** 54.5% fully supported claims, 45.5% partial, 2/3 safety pass.

In [ ]:
narratives = load_jsonl("final_narratives.jsonl")
validation = load_csv("validation_reports.csv")
claims = load_csv("claim_level_faithfulness.csv")

REQUIRED = ["case_id", "screening_summary", "model_explanation", "evidence_grounding",
            "recommended_next_step", "limitations", "citations", "safety_status"]
schema_ok = all(all(f in n for f in REQUIRED) for n in narratives)

disclaimer_rate = sum(
    1 for n in narratives if "not a clinical diagnosis" in json.dumps(n).lower()
) / len(narratives)
safety_pass_rate = (validation["overall_safety"] == "pass").mean()

summary_cases = pd.DataFrame([
    {"Metric": "Cases processed", "Value": len(narratives)},
    {"Metric": "Schema valid (all fields)", "Value": schema_ok},
    {"Metric": "Disclaimer rate", "Value": f"{disclaimer_rate:.0%}"},
    {"Metric": "Safety pass rate", "Value": f"{safety_pass_rate:.0%} ({validation['overall_safety'].eq('pass').sum()}/{len(validation)})"},
    {"Metric": "Total verified claims", "Value": len(claims)},
    {"Metric": "Unsupported/contradicted", "Value": claims["support_status"].isin(["unsupported", "contradicted"]).sum()},
])
display(summary_cases.style.hide(axis="index").set_caption("Table 3 — Three-case advisory evaluation"))
display(validation)

if len(claims):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    claims["support_status"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title("Claim support distribution")
    claims.groupby("case_id")["support_status"].value_counts().unstack(fill_value=0).plot(
        kind="bar", stacked=True, ax=axes[1], colormap="Set2"
    )
    axes[1].set_title("Claims by case")
    plt.tight_layout()
    plt.show()

show_figure("fig03_claim_support_distribution.png")
show_figure("fig04_safety_violation_types.png")

In [ ]:
if len(claims):
    display(claims[["case_id", "support_status", "claim", "cited_chunk_id"]].style.set_caption("Claim-level faithfulness"))

## 6. System comparison — B1 / B2 / B3

| System | Pipeline |
|--------|----------|
| **B1** | Retrieve → single-shot report (no evidence agent) |
| **B2** | Retrieve → evidence agent → report (no safety/citation tools) |
| **B3** | Full lean pipeline (primary) |

Evaluated on **3 cases** + **20-question** gold-chunk benchmark subset (B1 for benchmark).

In [ ]:
sys_cmp = load_csv("system_comparison.csv")
sys_summary_path = RESULTS / "system_comparison_summary.csv"
sys_summary = pd.read_csv(sys_summary_path, index_col=0) if sys_summary_path.exists() else None

cases_only = sys_cmp[sys_cmp["source"] == "case"] if "source" in sys_cmp.columns else sys_cmp
bench_only = sys_cmp[sys_cmp["source"] == "benchmark"] if "source" in sys_cmp.columns else pd.DataFrame()

display(cases_only.groupby("system").mean(numeric_only=True).round(3)
          .style.set_caption("Table 4 — System comparison (3 cases, mean metrics)"))

if len(bench_only):
    b1_bench = bench_only[bench_only["system"] == "B1"]
    print(f"Benchmark subset (B1): n={len(b1_bench)} questions")
    print(f"  Mean keyword recall: {b1_bench['keyword_recall'].mean():.3f}")
    print(f"  Safety pass rate: {b1_bench['safety_pass'].mean():.0%}")
    print(f"  Disclaimer rate: {b1_bench['disclaimer_present'].mean():.0%}")

if sys_summary is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    cols = [c for c in ["safety_pass", "claim_support_rate", "disclaimer_present", "keyword_recall"] if c in sys_summary.columns]
    sys_summary[cols].plot(kind="bar", ax=ax, color=PALETTE[:len(cols)])
    ax.set_title("System comparison B1–B3 (aggregate)")
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

show_figure("fig05_system_comparison_b1_b3.png")

## 7. Phase 2 vs Phase 3 baseline comparison

In [ ]:
bm25_p5 = retrieval_summary.loc[retrieval_summary["retriever"] == "bm25", "precision@5"].iloc[0]
supported_rate = (claims["support_status"] == "supported").mean() if len(claims) else np.nan
partial_rate = (claims["support_status"] == "partially_supported").mean() if len(claims) else np.nan

baseline_cmp = pd.DataFrame([
    {"Metric": "BM25 P@5 (gold 100Q)", "Phase 2": 0.128, "Phase 3 (this)": bm25_p5},
    {"Metric": "Hybrid P@5 (gold 100Q)", "Phase 2": 0.088, "Phase 3 (this)": retrieval_summary.loc[retrieval_summary["retriever"].str.contains("hybrid"), "precision@5"].iloc[0] if len(retrieval_summary) > 1 else np.nan},
    {"Metric": "Safety pass (3 cases)", "Phase 2": "2/3", "Phase 3 (this)": f"{validation['overall_safety'].eq('pass').sum()}/3"},
    {"Metric": "Disclaimer rate (3 cases)", "Phase 2": "partial", "Phase 3 (this)": f"{disclaimer_rate:.0%}"},
    {"Metric": "Claim supported rate", "Phase 2": "54.5%", "Phase 3 (this)": f"{supported_rate:.0%}" if pd.notna(supported_rate) else "—"},
    {"Metric": "Claim partial rate", "Phase 2": "45.5%", "Phase 3 (this)": f"{partial_rate:.0%}" if pd.notna(partial_rate) else "—"},
    {"Metric": "Unsupported/contradicted (final)", "Phase 2": "present", "Phase 3 (this)": claims["support_status"].isin(["unsupported", "contradicted"]).sum()},
    {"Metric": "100Q keyword recall@5 (Phase 2 Ollama eval)", "Phase 2": 0.157, "Phase 3 (this)": f"{bench_only[bench_only['system']=='B1']['keyword_recall'].mean():.3f}" if len(bench_only) else "—"},
    {"Metric": "100Q regex safety (Phase 2)", "Phase 2": "61%", "Phase 3 (this)": f"{bench_only[bench_only['system']=='B1']['safety_pass'].mean():.0%}" if len(bench_only) else "—"},
])
display(baseline_cmp.style.hide(axis="index").set_caption("Table 5 — Baseline comparison"))

## 8. Sample advisory report outputs

In [ ]:
for n in narratives:
    print("=" * 72)
    print(f"Case: {n['case_id']}")
    print(f"Summary: {str(n.get('screening_summary', ''))[:200]}")
    print(f"Next step: {str(n.get('recommended_next_step', ''))[:250]}")
    print(f"Citations: {len(n.get('citations') or [])}")
    lim = str(n.get('limitations', ''))
    if 'clinical diagnosis' in lim.lower():
        print("Disclaimer: present")
    print()

## 9. Hypotheses & limitations

**H1:** Evidence agent improves claim support vs B1 — compare `claim_support_rate` in Table 4.  
**H2:** Safety tools improve disclaimer rate — B3 vs B1/B2 on cases.  
**H3:** Hybrid ≥ BM25 on Recall@10 — see retrieval table (requires full llama3 FAISS index for true hybrid).

**Limitations:** single Ollama model for generation and verification; retrieval quality bounded by Phase 2 corpus; expert review mandatory before any real-world use.

In [ ]:
limitations_path = PAPER / "limitations.md"
if limitations_path.exists():
    from IPython.display import Markdown
    display(Markdown(limitations_path.read_text(encoding="utf-8")))

## 10. Export — reload paper tables from Excel

In [ ]:
xlsx = PAPER / "tables_for_paper.xlsx"
if xlsx.exists():
    xl = pd.ExcelFile(xlsx)
    print("Sheets:", xl.sheet_names)
    for sheet in xl.sheet_names:
        print(f"\n--- {sheet} ---")
        display(pd.read_excel(xlsx, sheet_name=sheet).head(10))
else:
    print("Run: python -m local_agentic_xai_rag.visualization.build_all_figures")

## Appendix — Regenerate results (optional)

Uncomment and run to re-execute the full pipeline (requires Ollama `llama3` running).

In [ ]:
# !python -m local_agentic_xai_rag.main --all-cases --bm25-only --fresh
# !python -m local_agentic_xai_rag.evaluation.retrieval_eval --limit 100 --bm25-only-hybrid
# !python -m local_agentic_xai_rag.evaluation.system_comparison --bm25-only --benchmark-limit 20
# !python -m local_agentic_xai_rag.visualization.build_all_figures
# !python -m local_agentic_xai_rag.evaluation.acceptance